# 22 — Incremental Parquet Pipelines

Demonstrates `ParquetPipeline.incremental()` for end-to-end incremental materialization of SQL sources to parquet datasets.

Key APIs showcased:
- `ParquetPipeline.incremental()` — factory that wires watermark tracking into materialize
- Automatic watermark commit after each `materialize()`
- Delta-only reads on subsequent runs
- `reload=True` to read back the freshly-written parquet

In [1]:
import datetime as dt
import os
from pathlib import Path
from tempfile import TemporaryDirectory

from sqlalchemy import Date, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import (
    DataHelper,
    FileWatermarkStore,
    ParquetPipeline,
)

In [2]:
class Base(DeclarativeBase):
    pass


class Event(Base):
    __tablename__ = "events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))

In [3]:
tmp = TemporaryDirectory()
root = Path(tmp.name)
db_path = root / "events.db"
sqlite_dsn = f"sqlite:///{db_path}"

engine = create_engine(sqlite_dsn)
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([
        Event(id=1, event_date=dt.date(2026, 5, 1), status="active"),
        Event(id=2, event_date=dt.date(2026, 5, 2), status="inactive"),
        Event(id=3, event_date=dt.date(2026, 5, 3), status="active"),
        Event(id=4, event_date=dt.date(2026, 5, 4), status="active"),
        Event(id=5, event_date=dt.date(2026, 5, 5), status="inactive"),
    ])
    session.commit()
engine.dispose()

print(f"Seeded 5 rows at {sqlite_dsn}")

Seeded 5 rows at sqlite:////var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmpvxcy9ujq/events.db


## Create an incremental pipeline

`ParquetPipeline.incremental()` wraps a `DataHelper` source so that every `materialize()` call pulls only rows newer than the last materialized watermark.

In [4]:
helper = DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="events",
)

pipeline = ParquetPipeline.incremental(
    helper,
    {
        "storage_path": str(root / "events_dataset"),
        "project_root": root,
    },
    watermark_field="event_date",
    watermark_source="events_pipeline",
    date_field="event_date",
    watermark_store=FileWatermarkStore(path=str(root / ".watermarks.json")),
)

print(f"Pipeline created — watermark_field={pipeline._watermark_field}, watermark_source={pipeline._watermark_source}")

Pipeline created — watermark_field=event_date, watermark_source=events_pipeline


## First materialize

No watermark exists yet, so all 5 rows are loaded and written to parquet. The watermark is automatically committed from the reloaded parquet frame.

In [5]:
r1 = pipeline.materialize(reload=True)
frame1 = r1.frame.compute().sort_values("id").reset_index(drop=True) if r1.frame is not None else None

print(f"path:    {r1.path}")
print(f"reloaded:{r1.reloaded}")
print(f"rows:    {len(frame1) if frame1 is not None else 0}")
frame1

path:    /var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmpvxcy9ujq/events_dataset
reloaded:True
rows:    5


/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: worker_connection_env_var is not set; falling back to raw DSN. Credentials may be visible in scheduler logs.
  self._worker_config = WorkerSqlConfig.from_database_config(config)


,id,event_date,status,partition_date
0,1,2026-05-01 00:00:00+00:00,active,2026-05-01
1,2,2026-05-02 00:00:00+00:00,inactive,2026-05-02
2,3,2026-05-03 00:00:00+00:00,active,2026-05-03
3,4,2026-05-04 00:00:00+00:00,active,2026-05-04
4,5,2026-05-05 00:00:00+00:00,inactive,2026-05-05


## Second materialize — no new data

Because the watermark has been committed, the pipeline now loads only rows with `event_date > 2026-05-05`. None exist, so the parquet write is empty.

In [6]:
r2 = pipeline.materialize(reload=True)
frame2 = r2.frame.compute().sort_values("id").reset_index(drop=True) if r2.frame is not None else None

print(f"reloaded:{r2.reloaded}")
print(f"rows:    {len(r2.frame) if r2.frame is not None else 0}")

reloaded:False
rows:    0


/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: worker_connection_env_var is not set; falling back to raw DSN. Credentials may be visible in scheduler logs.
  self._worker_config = WorkerSqlConfig.from_database_config(config)


## Third materialize — new rows in source

Insert two new events into the source table, then materialize again. Only the new rows are loaded and written.

In [7]:
engine = create_engine(sqlite_dsn)
with Session(engine) as session:
    session.add_all([
        Event(id=6, event_date=dt.date(2026, 5, 6), status="active"),
        Event(id=7, event_date=dt.date(2026, 5, 7), status="active"),
    ])
    session.commit()
engine.dispose()
print("Inserted 2 new rows.")

Inserted 2 new rows.


In [8]:
r3 = pipeline.materialize(reload=True)
frame3 = r3.frame.compute().sort_values("id").reset_index(drop=True)

print(f"rows: {len(frame3)}")
frame3

rows: 2


/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: worker_connection_env_var is not set; falling back to raw DSN. Credentials may be visible in scheduler logs.
  self._worker_config = WorkerSqlConfig.from_database_config(config)


,id,event_date,status,partition_date
0,6,2026-05-06 00:00:00+00:00,active,2026-05-06
1,7,2026-05-07 00:00:00+00:00,active,2026-05-07


## Cleanup

In [9]:
helper.close()
tmp.cleanup()
print("Cleaned up.")

Cleaned up.
